# 🫀 실험 2 — P파 분할: **재조합이 아니라 새 정보를 넣는다**

**MedKOS / `notebooks/exp2_pwave_delineation.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 실험 1′이 알려준 것

| | |
|---|---|
| 결과 | B3R − B3 = **+0.0049** [−0.0254, +0.0321] · CI 반폭 0.0287 → **기각(확정)** |
| 왜 실패했나 | **보상지수는 RR의 재조합**이다. B3가 이미 RR을 갖고 있으니 새 정보가 아니었다 |
| 그래서 | `"재조합은 안 오른다, 새 정보만 오른다"` — 선행 트랙 13건 음성결과의 교훈이 물리 특징에도 적용됐다 |

**이번 실험은 그 원칙을 정면으로 시험한다.** P파의 위치·형태는 **RR에서 유도할 수 없다.**
진짜 새 관측이다. 선행 트랙도 `FINDINGS.md`에서 "P파 벡터-에너지 특징만이 유일하게
label-free 판별 정보를 실제로 더했다"고 보고했고, 동시에 **"avg-pool CNN이 작은 P파를
QRS에 묻어버린다"** 는 한계를 지적했다.

→ **그렇다면 P파를 묻지 말고 명시적으로 분할해서 꺼내면 되지 않는가?** 그것이 실험 2다.

---

## 설계

```
LUDB (200명 · 심장전문의가 P/QRS/T 경계를 직접 주석)
   │  1D U-Net 분할기 학습  (lead I + lead II 둘 다로 학습 → 축에 강건)
   ▼
Icentia11k 코호트 (실험 1′에서 이미 받아둔 캐시 재사용 — 재다운로드 없음)
   │  비트마다 P파 6개 특징 추출
   ▼
B3 (형태+RR)   vs   B3P (형태+RR+P파)      ← 사전등록 비교
```

**P파 특징 6개** (RR에서 유도 불가능한 것 위주)

| 특징 | 의미 |
|---|---|
| `p_prob` | P파 존재 확률 (분할기의 P 마스크 최대값) |
| `p_dur` | P 지속시간 |
| `p_amp` | P 진폭 |
| `p_area` | P 면적(적분) — 원 PPT의 "하부 적분값" |
| `pr_int` | P 시작 → R (PR 간격) |
| `p_dev` | **환자 자신의 정상 P 대비 편차** (label-free 개인화) |

`p_dev`가 핵심이다. 선행 트랙이 "환자 정상 기준의 P파 특징"을 강조했고, 라벨을 쓰지 않으므로
inter-patient에서 안전하다.

---

## 사전등록 (실행 전 고정)

```
주가설 : B3P − B3 > 0        (P파 분할이 RR 위에 정보를 더하는가)
주지표 : 환자단위 매크로 F1(N/S/V), 이소성 보유 환자만
비교   : 대응 부트스트랩 B=2000, 환자 리샘플
판정   : CI가 0을 벗어나면 확증 / 포함하고 반폭<0.03이면 기각(확정) / 그 외 미결
저울   : seed 3개 확률 평균 앙상블
금지   : 결과를 보고 특징·가중치·에폭을 바꾸지 않는다
분할   : 실험 1′과 동일한 환자 분할(SEED0 고정) — arm 비교가 오염되지 않도록
```

## 게이트 두 개 (비싼 학습 전에 싸게 거른다)

| 게이트 | 언제 | 통과 기준 | 실패하면 |
|---|---|---|---|
| **G1** 분할기 품질 | CELL 5 | LUDB held-out에서 **P파 검출 F1 ≥ 0.70** | 깨끗한 임상 데이터에서도 P를 못 찾으면 노이즈 많은 웨어러블에선 가망 없음 → 중단 |
| **G2** 특징 판별력 | CELL 7 | N↔S 분리 **\|Cohen's d\| ≥ 0.3** (하나 이상의 P 특징에서) | 실험 1′에서 d=0.283이 +0.005로 이어졌다. 여기서 낮으면 학습해도 안 오른다 |

---

## ⚠️ 미리 아는 약점 (정직하게)

1. **도메인 갭**: LUDB는 임상 12유도·500 Hz·깨끗함, Icentia는 웨어러블·250 Hz·노이즈 많음.
   분할기가 전이되지 않을 수 있다 — 그게 G1·G2로 걸러진다.
2. **Icentia는 modified lead I** 이다. P파는 lead II에서 가장 크다. 그래서 분할기를
   **lead I과 II 둘 다로 학습**시켜 축에 강건하게 만든다(그래도 lead II보다 불리하다).
3. `pr_int`는 R 위치에 의존하므로 RR과 부분적으로 결합돼 있다. 나머지 5개는 형태 기반이라
   RR과 독립적이다.


In [ ]:
# CELL 1 — Drive 프로젝트 + 아티팩트 헬퍼 (실험 1′이 만들어둔 lib 재사용)
!pip -q install wfdb

import os, sys, json, time, re, numpy as np
from collections import Counter

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님 — 로컬 대체:", e); DRIVE_ROOT = "/content"

PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
LIB = os.path.join(PROJECT, "lib")
sys.path.insert(0, LIB)
try:
    from medkos_run import MedKOSRun
    print("✅ lib/medkos_run.py 재사용 (실험 1′이 남긴 것)")
except Exception as e:
    raise SystemExit(f"❌ {LIB}/medkos_run.py 가 없습니다. exp1 노트북 CELL 1을 먼저 실행하세요. ({e})")

QUICK = True          # True: LUDB 분할기까지만 빠르게 / False: 본 실행
SEED0 = 20260731      # ★ 실험 1′과 동일 — 환자 분할을 그대로 써야 비교가 오염되지 않음
FS, W_PRE, W_POST = 250, 100, 150
CLASSES, TEST_FRAC = ["N", "S", "V"], 0.35
N_SEEDS, EPOCHS = (2, 20) if QUICK else (3, 30)
N_UNDERSAMPLE = 10

CONFIG = dict(exp="exp2_pwave_delineation", quest="ailab-2026-0015",
              hypothesis="B3P - B3 > 0 (P파 분할이 RR 위에 새 정보를 더하는가)",
              delineator="1D U-Net on LUDB (lead I+II, 250Hz, R-centered)",
              p_features=["p_prob", "p_dur", "p_amp", "p_area", "pr_int", "p_dev"],
              gates={"G1_ludb_P_f1": 0.70, "G2_cohens_d": 0.30},
              fs=FS, w_pre=W_PRE, w_post=W_POST, seed0=SEED0, test_frac=TEST_FRAC,
              n_seeds=N_SEEDS, epochs=EPOCHS, n_undersample=N_UNDERSAMPLE, quick=QUICK,
              boot=2000)
np.random.seed(SEED0)
# 그림 한글 깨짐 방지 (Colab)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"],
                       capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as _e:
    print("한글 폰트 설정 생략:", _e)

run = MedKOSRun("exp2_pwave", CONFIG, project=PROJECT)
run.log(f"QUICK={QUICK} · seed {N_SEEDS} · epoch {EPOCHS}")

### CELL 2 — LUDB 경로·주석 확인 (★ 여기서 한 번 멈추세요)

LUDB 주석은 **유도별로 파일이 따로**입니다(`1.i`, `1.ii`, …). 심볼은 파형 피크(`p`/`N`/`t`)와
경계(`(`, `)`)로 이루어져 있습니다. 경로 규칙을 추측하지 않고 확인부터 합니다.

In [ ]:
# CELL 2 — LUDB 구조 탐색
import wfdb

LUDB_CANDIDATES = ["ludb/1.0.1/data", "ludb/1.0.1", "ludb/1.0.0/data", "ludb"]
LUDB_DIR, probe = None, {}
for d in LUDB_CANDIDATES:
    try:
        rec = wfdb.rdrecord("1", pn_dir=d)
        ann = wfdb.rdann("1", "ii", pn_dir=d)
        LUDB_DIR = d
        probe = {"dir": d, "fs": rec.fs, "sig": rec.sig_name,
                 "shape": list(rec.p_signal.shape),
                 "symbols": Counter(ann.symbol).most_common()}
        run.log(f"✅ LUDB_DIR = {d}")
        run.log(f"   fs={rec.fs} sig={rec.sig_name} shape={rec.p_signal.shape}")
        run.log(f"   주석 심볼(lead ii) = {probe['symbols']}")
        run.log(f"   앞 12개 (sample, symbol) = "
                f"{list(zip(ann.sample[:12], ann.symbol[:12]))}")
        break
    except Exception as e:
        run.log(f"❌ {d}: {type(e).__name__}: {str(e)[:90]}")

run.save_json("ludb_probe", probe)
run.log("\n확인: ① 심볼에 '(' , 'p'/'N'/'t' , ')' 가 보이는가")
run.log("      ② sig_name 에 i, ii 가 있는가 (대소문자 주의)")
run.log("      ③ 전부 ❌면 PhysioNet ludb 페이지의 Files 구조를 알려주세요")

In [ ]:
# CELL 3 — LUDB → R중심 비트 창 + P/QRS/T 마스크 (250Hz로 리샘플)
from scipy.signal import resample_poly

WAVE = {"p": 1, "N": 2, "t": 3}          # 배경 0
LEADS_TRAIN = ["ii", "i"]                # ★ 두 유도로 학습 → 축에 강건

def ludb_record(rid, lead):
    rec = wfdb.rdrecord(str(rid), pn_dir=LUDB_DIR)
    names = [n.strip().lower() for n in rec.sig_name]
    if lead not in names:
        return None
    sig = rec.p_signal[:, names.index(lead)].astype("float64")
    ann = wfdb.rdann(str(rid), lead, pn_dir=LUDB_DIR)
    # 500Hz → 250Hz
    if rec.fs != FS:
        g = int(round(rec.fs / FS))
        sig = resample_poly(sig, 1, g)
        samp = (np.asarray(ann.sample) / g).round().astype(int)
    else:
        samp = np.asarray(ann.sample)
    sym = list(ann.symbol)

    mask = np.zeros(len(sig), "int8")
    rpeaks = []
    i = 0
    while i < len(sym) - 2:
        if sym[i] == "(" and sym[i + 1] in WAVE and sym[i + 2] == ")":
            a, b = max(0, samp[i]), min(len(sig), samp[i + 2] + 1)
            mask[a:b] = WAVE[sym[i + 1]]
            if sym[i + 1] == "N":
                rpeaks.append(int(samp[i + 1]))
            i += 3
        else:
            i += 1
    return sig, mask, np.array(rpeaks, int)

def robust(x):
    m = np.median(x); q = np.percentile(x, 75) - np.percentile(x, 25)
    return ((x - m) / (q + 1e-6)).astype("float32")

CACHE = run.data("ludb_beats_250hz.npz")
if os.path.exists(CACHE):
    d = np.load(CACHE); Xl, Ml, Pl = d["X"], d["M"], d["pid"]
    run.log(f"LUDB 캐시 재사용: {Xl.shape}")
else:
    Xs, Ms, Ps = [], [], []
    fails = 0
    for rid in range(1, 201):
        for lead in LEADS_TRAIN:
            try:
                out = ludb_record(rid, lead)
            except Exception:
                fails += 1; continue
            if out is None:
                continue
            sig, mask, rp = out
            sig = robust(sig)
            for r in rp:
                if r - W_PRE < 0 or r + W_POST > len(sig):
                    continue
                Xs.append(sig[r - W_PRE:r + W_POST])
                Ms.append(mask[r - W_PRE:r + W_POST])
                Ps.append(rid)
        if rid % 50 == 0:
            run.log(f"  LUDB {rid}/200 ({len(Xs)} beats)")
    Xl = np.array(Xs, "float32"); Ml = np.array(Ms, "int8"); Pl = np.array(Ps, int)
    np.savez_compressed(CACHE, X=Xl, M=Ml, pid=Pl)
    run.log(f"저장: {CACHE}  (실패 {fails}건)")

run.log(f"\nLUDB 비트 {len(Xl):,}개 · 환자 {len(np.unique(Pl))}명")
frac = {k: float((Ml == v).mean()) for k, v in
        {"배경": 0, "P": 1, "QRS": 2, "T": 3}.items()}
run.log(f"마스크 비율 {({k: round(v,3) for k,v in frac.items()})}")
run.save_json("ludb_stats", {"n_beats": int(len(Xl)),
                             "n_patients": int(len(np.unique(Pl))), "mask_frac": frac})
if frac["P"] < 0.01:
    run.log("⛔ P 마스크가 거의 없습니다 — CELL 2의 심볼 매핑을 확인하세요")

In [ ]:
# CELL 4 — 1D U-Net 분할기 학습 (LUDB 환자 단위 분리)
import tensorflow as tf
from tensorflow.keras import layers, models

L = 256                                   # 250 → 256 패딩
def pad(A):
    return np.pad(A, ((0, 0), (0, L - A.shape[1])), mode="edge")

lp = np.unique(Pl); rs = np.random.RandomState(SEED0); rs.shuffle(lp)
lv = set(lp[:max(20, len(lp) // 5)].tolist())          # held-out 환자
vm = np.isin(Pl, list(lv))
Xtr, Mtr = pad(Xl[~vm])[..., None], pad(Ml[~vm].astype("int32"))
Xva, Mva = pad(Xl[vm])[..., None], pad(Ml[vm].astype("int32"))
run.log(f"분할기 학습 {len(Xtr)} beats / 검증 {len(Xva)} beats "
        f"(환자 {len(lp)-len(lv)} / {len(lv)})")

def unet(seed):
    tf.keras.utils.set_random_seed(seed)
    inp = layers.Input((L, 1))
    def blk(x, f):
        x = layers.Conv1D(f, 9, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        return layers.Conv1D(f, 9, padding="same", activation="relu")(x)
    c1 = blk(inp, 24); p1 = layers.MaxPooling1D(2)(c1)
    c2 = blk(p1, 48);  p2 = layers.MaxPooling1D(2)(c2)
    c3 = blk(p2, 96)
    u2 = layers.Concatenate()([layers.UpSampling1D(2)(c3), c2]); d2 = blk(u2, 48)
    u1 = layers.Concatenate()([layers.UpSampling1D(2)(d2), c1]); d1 = blk(u1, 24)
    out = layers.Conv1D(4, 1, activation="softmax")(d1)
    m = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

SEG = unet(SEED0)
h = SEG.fit(Xtr, Mtr, validation_data=(Xva, Mva),
            epochs=30 if not QUICK else 20, batch_size=64, verbose=0)
run.log(f"분할기 loss {h.history['loss'][0]:.3f}→{h.history['loss'][-1]:.3f} | "
        f"val {h.history['val_loss'][0]:.3f}→{h.history['val_loss'][-1]:.3f}")
run.save_model(SEG, "delineator")

### CELL 5 — 게이트 G1: 분할기가 깨끗한 데이터에서 P파를 찾는가

**깨끗한 임상 12유도에서도 P를 못 찾으면, 노이즈 많은 웨어러블에서는 가망이 없습니다.**
여기서 걸러야 뒤의 비싼 학습을 낭비하지 않습니다.

In [ ]:
# CELL 5 — G1: LUDB held-out P파 검출 성능
pv = SEG.predict(Xva, batch_size=256, verbose=0).argmax(-1)
g1 = {}
for name, cls in (("P", 1), ("QRS", 2), ("T", 3)):
    t, p = (Mva == cls), (pv == cls)
    inter = (t & p).sum()
    dice = 2 * inter / (t.sum() + p.sum() + 1e-9)              # 샘플 단위 Dice
    # 비트 단위: 참 마스크와 예측이 겹치면 검출 성공
    hit = ((t & p).sum(1) > 0.3 * np.maximum(t.sum(1), 1))
    has = t.sum(1) > 0
    det = float(hit[has].mean()) if has.any() else 0.0
    g1[name] = {"dice": float(dice), "beat_detection": det, "n_beats_with": int(has.sum())}
    run.log(f"  {name:4s} Dice={dice:.3f}  비트검출률={det:.3f}  (해당 비트 {int(has.sum())})")

P_F1 = g1["P"]["dice"]
run.save_json("gate1_delineator", g1)
run.log("")
if P_F1 >= CONFIG["gates"]["G1_ludb_P_f1"]:
    run.log(f"✅ G1 통과 — P Dice {P_F1:.3f} ≥ {CONFIG['gates']['G1_ludb_P_f1']}")
else:
    run.log("⛔" * 30)
    run.log(f"⛔ G1 실패 — P Dice {P_F1:.3f} < {CONFIG['gates']['G1_ludb_P_f1']}")
    run.log("⛔ 깨끗한 임상 데이터에서도 P를 못 찾습니다. 웨어러블 적용은 무의미합니다.")
    run.log("⛔ 여기서 멈추고 알려주세요 (epoch↑ / 창 길이↑ / lead II 단독 학습 등 검토).")
    run.log("⛔" * 30)

# 눈으로 확인 — 예측 마스크 3개
import matplotlib.pyplot as plt
idx = np.where(Mva.sum(1) > 0)[0][:3]
fig, ax = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for k, i in enumerate(idx):
    ax[k].plot(Xva[i, :, 0], lw=1, color="#222")
    for cls, col in ((1, "#c0392b"), (2, "#1c4780"), (3, "#2e7d4f")):
        ax[k].fill_between(range(L), -3, 3, where=(pv[i] == cls), alpha=.18, color=col)
    ax[k].set_ylim(-4, 4); ax[k].set_ylabel(f"beat {i}")
ax[0].set_title("LUDB held-out — 예측 마스크 (빨강 P · 파랑 QRS · 초록 T)")
plt.tight_layout(); run.save_fig("gate1_delineation", fig); plt.show()

In [ ]:
# CELL 6 — 실험 1′ 코호트 재사용 + P파 특징 추출
# 실험 1′이 Drive data/ 에 남긴 캐시를 그대로 쓴다(재다운로드 없음).
cands = sorted([f for f in os.listdir(run.data_dir) if f.startswith("icentia_v2_")])
if not cands:
    raise SystemExit("❌ 실험 1′ 코호트 캐시가 없습니다. exp1 노트북을 먼저 돌리세요.")
COH = run.data(cands[-1])
d = np.load(COH); Xb, yb, tb, pidb = d["X"], d["y"], d["t"], d["pid"]
run.log(f"코호트 재사용: {cands[-1]}  비트 {len(yb):,} · 환자 {len(np.unique(pidb))}")

Xi = np.clip(np.nan_to_num(Xb.astype("float32")), -20, 20)
seg = SEG.predict(pad(Xi)[..., None], batch_size=512, verbose=0)   # [n, 256, 4]
pmask = seg[:, :W_PRE + W_POST, 1]                                  # P 확률
sigm = Xi

# ── 비트별 P파 특징 6개 ──
thr = 0.5
on = pmask > thr
p_prob = pmask.max(1)
p_dur = on.sum(1) / FS
p_amp = np.where(on.any(1), (sigm * on).max(1), 0.0)
p_area = (sigm * on).sum(1) / FS
first = np.where(on.any(1), on.argmax(1), W_PRE)
pr_int = (W_PRE - first) / FS
P = np.stack([p_prob, p_dur, p_amp, p_area, pr_int], 1).astype("float32")

# p_dev: 환자 자신의 '정상 P' 대비 편차 (라벨 미사용 — 환자 중앙값 기준)
p_dev = np.zeros(len(yb), "float32")
for pid in np.unique(pidb):
    m = pidb == pid
    med = np.median(P[m], 0); iqr = np.percentile(P[m], 75, 0) - np.percentile(P[m], 25, 0)
    p_dev[m] = np.abs((P[m] - med) / (iqr + 1e-3)).mean(1)
F_P = np.concatenate([P, p_dev[:, None]], 1)
F_P = np.clip(np.nan_to_num(F_P), [0, 0, -5, -5, -0.1, 0], [1, 0.4, 20, 20, 0.5, 20])
run.save_npy("features_P", F_P)
run.log(f"P 특징 {F_P.shape}  범위 min={F_P.min(0).round(3).tolist()}")
run.log(f"                     max={F_P.max(0).round(3).tolist()}")
run.log(f"P 검출률(p_prob>0.5): 전체 {float((p_prob>0.5).mean()):.1%}  "
        f"N {float((p_prob[yb==0]>0.5).mean()):.1%}  "
        f"S {float((p_prob[yb==1]>0.5).mean()):.1%}  "
        f"V {float((p_prob[yb==2]>0.5).mean()):.1%}")
run.log("  ※ V(심실조기)에서 P 검출률이 낮게 나오면 분할기가 제대로 도는 신호입니다")

### CELL 7 — 게이트 G2: P 특징이 실제로 N과 S를 가르는가

실험 1′에서 사전점검 `d = 0.283`이 최종 `+0.005`로 이어졌습니다. **여기서 낮으면 학습해도
안 오릅니다.** 비싼 학습 전에 2분 만에 판단합니다.

In [ ]:
# CELL 7 — G2: P 특징의 N↔S 판별력 (사전점검)
FEATNAMES = CONFIG["p_features"]
g2, best = {}, 0.0
run.log("특징별 N↔S 분리 (Cohen's d)")
for j, nm in enumerate(FEATNAMES):
    a, b = F_P[yb == 0, j], F_P[yb == 1, j]
    if len(b) < 20:
        continue
    dd = float((a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2 + 1e-9))
    g2[nm] = {"cohens_d": dd, "mean_N": float(a.mean()), "mean_S": float(b.mean())}
    best = max(best, abs(dd))
    run.log(f"  {nm:8s} d={dd:+.3f}   N={a.mean():.3f}  S={b.mean():.3f}")
run.save_json("gate2_pfeatures", {"per_feature": g2, "best_abs_d": best})
run.log(f"\n최대 |d| = {best:.3f}  (기준 {CONFIG['gates']['G2_cohens_d']})")
if best >= CONFIG["gates"]["G2_cohens_d"]:
    run.log("✅ G2 통과 — 학습으로 짜낼 신호가 있습니다")
else:
    run.log("⛔" * 30)
    run.log(f"⛔ G2 실패 — 최대 |d|={best:.3f}. 실험 1′(d=0.283 → +0.005)의 전례상")
    run.log("⛔ 학습해도 이득이 없을 가능성이 높습니다. 그래도 돌릴지 판단이 필요합니다.")
    run.log("⛔ (원인 후보: LUDB→Icentia 도메인 갭, modified lead I에서 P파가 작음)")
    run.log("⛔" * 30)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for k, j in enumerate([0, 2, 5]):
    for c, col, nm in ((0, "#888", "N"), (1, "#c0392b", "S"), (2, "#1c4780", "V")):
        v = F_P[yb == c, j]
        if len(v) > 10:
            ax[k].hist(v, bins=50, alpha=.5, density=True, color=col, label=nm)
    ax[k].set_title(FEATNAMES[j]); ax[k].legend(fontsize=8)
plt.tight_layout(); run.save_fig("gate2_pfeatures", fig); plt.show()

In [ ]:
# CELL 8 — arm 학습: B3(형태+RR) vs B3P(형태+RR+P)
#   실험 1′과 동일한 RR 특징·분할·언더샘플·가중치를 그대로 사용한다.
ALPHA, K_REF = 0.3, 10
def rhythm_rr(t):
    n = len(t); rr = np.diff(t)
    if len(rr) < 5:
        return None
    rr_ref = np.maximum([np.median(rr[max(0, i-K_REF):i+K_REF+1]) for i in range(len(rr))], 1e-3)
    out = []
    for k in range(n):
        i0, i1 = k - 1, k
        if i0 < 0 or i1 >= len(rr):
            out.append([1., 1., .8]); continue
        out.append([rr[i0]/rr_ref[i0], rr[i1]/rr_ref[i0], rr_ref[i0]])
    return np.array(out, "float32")

F_RR = np.zeros((len(yb), 3), "float32")
for pid in np.unique(pidb):
    m = pidb == pid
    a = rhythm_rr(tb[m])
    if a is not None:
        F_RR[m] = a
F_RR = np.clip(np.nan_to_num(F_RR), [0, 0, 0.2], [4, 4, 3.0])

pats = np.unique(pidb); rs = np.random.RandomState(SEED0); rs.shuffle(pats)
is_te = np.isin(pidb, pats[:int(len(pats) * TEST_FRAC)])
tr_idx = np.where(~is_te)[0]
_rs = np.random.RandomState(SEED0)
ect = tr_idx[yb[tr_idx] != 0]; nrm = tr_idx[yb[tr_idx] == 0]
keep = min(len(nrm), max(len(ect) * N_UNDERSAMPLE, 2000))
tr_idx = np.sort(np.concatenate([ect, _rs.choice(nrm, keep, replace=False)]))

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(3)}
    return {c: float(v / w[0]) for c, v in w.items()}
CW = auto_weights(yb[tr_idx])
SW = np.array([CW[int(c)] for c in yb[tr_idx]], "float32")
run.log(f"학습 {len(tr_idx):,}비트 · 테스트 환자 {int(len(np.unique(pidb[is_te])))}명")
run.log(f"가중 후 유효 개수 { {CLASSES[c]: int((yb[tr_idx]==c).sum()*CW[c]) for c in range(3)} }")

def feat_norm(F, tri):
    med = np.median(F[tri], 0)
    iqr = np.percentile(F[tri], 75, 0) - np.percentile(F[tri], 25, 0)
    return np.clip((F - med) / (iqr + 1e-3), -10, 10).astype("float32")

def build(nf, seed):
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((W_PRE + W_POST, 1)); x = si
    for f, k in ((32, 7), (64, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x); x = layers.Dense(64, activation="relu")(x)
    fi = layers.Input((nf,))
    g = layers.Dense(32, activation="relu")(fi); g = layers.Dense(32, activation="relu")(g)
    x = layers.Concatenate()([x, g]); x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    m = models.Model([si, fi], layers.Dense(3, activation="softmax")(x))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

ARMS = {"B3": F_RR, "B3P": np.concatenate([F_RR, F_P], 1)}
Xf = np.clip(np.nan_to_num(Xb.astype("float32")), -20, 20)[..., None]
probs, t0 = {}, time.time()
for arm, F in ARMS.items():
    c = run.load_arm(arm)
    if c is not None:
        probs[arm] = c; run.log(f"⏭ {arm} 이미 완료"); continue
    acc = np.zeros((int(is_te.sum()), 3))
    Fn = feat_norm(F, tr_idx)
    for s in range(N_SEEDS):
        m = build(F.shape[1], SEED0 + s)
        h = m.fit([Xf[tr_idx], Fn[tr_idx]], yb[tr_idx], epochs=EPOCHS,
                  batch_size=256, sample_weight=SW, verbose=0)
        pr = m.predict([Xf[is_te], Fn[is_te]], batch_size=1024, verbose=0)
        acc += pr
        run.log(f"  {arm} seed {s+1}/{N_SEEDS} ({time.time()-t0:.0f}s) "
                f"loss {h.history['loss'][0]:.3f}→{h.history['loss'][-1]:.3f} | "
                f"예측분포 {np.bincount(pr.argmax(1), minlength=3).tolist()}")
        tf.keras.backend.clear_session()
    probs[arm] = acc / N_SEEDS
    run.save_arm(arm, probs[arm])

In [ ]:
# CELL 9 — 평가 + 사전등록 판정 (실험 1′과 동일 프로토콜)
from sklearn.metrics import f1_score, confusion_matrix
y_te, pid_te = yb[is_te], pidb[is_te]; upa = np.unique(pid_te)
HAS_ECT = np.array([bool(((pid_te == p) & (y_te != 0)).any()) for p in upa])
run.log(f"테스트 환자 {len(upa)}명 중 이소성 보유 {HAS_ECT.sum()}명 ← 주지표")

def ppm(pred):
    out = []
    for p in upa:
        m = pid_te == p
        pres = np.unique(np.concatenate([y_te[m], pred[m]]))
        out.append(f1_score(y_te[m], pred[m], labels=pres, average="macro", zero_division=0))
    return np.array(out)

res = {}
for arm, pr in probs.items():
    pd_ = pr.argmax(1); a = ppm(pd_)
    res[arm] = {"pred": pd_, "pp": a[HAS_ECT] if HAS_ECT.any() else a,
                "macro": float((a[HAS_ECT] if HAS_ECT.any() else a).mean()),
                "macro_all": float(a.mean()),
                "f1": f1_score(y_te, pd_, average=None, labels=[0, 1, 2], zero_division=0),
                "cm": confusion_matrix(y_te, pd_, labels=[0, 1, 2]).tolist()}

run.log("=" * 78)
run.log(f"{'ARM':<22}{'매크로(이소성)':>15}{'매크로(전체)':>14}{'N':>8}{'S':>8}{'V':>8}")
run.log("=" * 78)
for a, r in res.items():
    f = r["f1"]
    run.log(f"{a:<22}{r['macro']:>15.4f}{r['macro_all']:>14.4f}"
            f"{f[0]:>8.3f}{f[1]:>8.3f}{f[2]:>8.3f}")
run.log("=" * 78)

COLLAPSED = [a for a, r in res.items() if r["f1"][1] == 0 and r["f1"][2] == 0]
if COLLAPSED:
    run.log(f"⛔ 붕괴한 arm: {COLLAPSED} — 아래 판정을 믿지 마세요")

def boot(a, b, B=CONFIG["boot"], seed=SEED0):
    rs = np.random.RandomState(seed); d = a - b
    bs = d[rs.randint(0, len(d), (B, len(d)))].mean(1)
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

dd, lo, hi = boot(res["B3P"]["pp"], res["B3"]["pp"])
half = (hi - lo) / 2
run.log(f"\n▶ 사전등록 주가설 — B3P − B3 (P파 분할이 더하는 정보)")
run.log(f"   Δ = {dd:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   반폭 {half:.4f}")
run.log(f"   [참고] 실험1′ B3R−B3 = +0.0049 [−0.0254, +0.0321] (기각)")

if COLLAPSED:
    verdict = f"판정 불가 — arm 붕괴({COLLAPSED})"
elif int(HAS_ECT.sum()) < 20:
    verdict = f"판정 불가 — 이소성 보유 테스트 환자 {int(HAS_ECT.sum())}명뿐"
elif (lo > 0 or hi < 0) and dd > 0:
    verdict = "확증 — P파 분할은 RR 위에 새 정보를 더한다 (재조합이 아님)"
elif (lo > 0 or hi < 0) and dd < 0:
    verdict = "역효과 — P 특징이 오히려 해롭다(도메인 갭·특징 잡음 의심)"
elif half < 0.03:
    verdict = "기각(확정) — P파 분할도 이득 없음. 단일유도 웨어러블에서 P는 회수 불가에 가깝다"
else:
    verdict = "미결 — 검정력 부족. QUICK=False 또는 환자 수 확대"
run.log(f"   → {verdict}")

run.save_json("evaluation", {
    "arms": {a: {"macro_f1": r["macro"], "macro_f1_all": r["macro_all"],
                 "f1": {CLASSES[i]: float(r["f1"][i]) for i in range(3)},
                 "cm": r["cm"]} for a, r in res.items()},
    "B3P_minus_B3": {"delta": dd, "ci": [lo, hi], "halfwidth": half},
    "gate1_P_dice": P_F1, "gate2_best_abs_d": best,
    "n_patients_with_ectopy": int(HAS_ECT.sum()), "collapsed": COLLAPSED,
    "verdict": verdict})

result = {"week": 2, "exp_id": "exp2_pwave", "quest": "ailab-2026-0015",
          "task": "P파 분할로 새 정보 추가 (실험2)", "split": "inter",
          "metric": "macro_f1", "value": round(res["B3P"]["macro"], 4),
          "passed": bool((lo > 0) and dd > 0 and not COLLAPSED),
          "date": time.strftime("%Y-%m-%d"),
          "arms": {a: round(r["macro"], 4) for a, r in res.items()},
          "B3P_minus_B3": {"delta": dd, "ci": [lo, hi]},
          "gate1_P_dice": P_F1, "gate2_best_abs_d": best, "verdict": verdict,
          "summary": f"B3P-B3 {dd:+.4f} [{lo:+.4f},{hi:+.4f}] G1={P_F1:.2f} G2={best:.2f} → {verdict.split(' —')[0]}"}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp2_pwave_delineation.ipynb \\
      --quest ailab-2026-0015 --step "exp2-pwave-delineation" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

| B3P − B3 | 뜻 | 다음 |
|---|---|---|
| **0을 벗어나고 +** | **새 정보 가설 확증.** "재조합은 안 되고 새 관측은 된다"가 실증됨 | P파 축을 Finding Head의 정식 입력으로. 실험 10(유도 확장)으로 |
| **0 포함 · 반폭<0.03** | 단일유도 웨어러블에서 **P는 회수 불가에 가깝다** | 중요한 음성 결과. → 유도 확장(실험 10)이 유일한 남은 정보원이 됨 |
| **G1 실패** | 분할기가 깨끗한 데이터에서도 P를 못 찾음 | 학습 조건 문제 — 실험 자체는 아직 판정 안 됨 |
| **G2 실패, 그래도 돌림** | 특징에 신호가 없음 | 도메인 갭(LUDB 임상 → Icentia 웨어러블)이 원인일 가능성 |

**어느 쪽이든 실험 10(유도 ablation)으로 이어집니다.** P가 회수되면 "단일유도로 어디까지"의
상한이 올라가고, 회수되지 않으면 "유도를 늘리는 것 말고 답이 없다"가 증명됩니다.
